In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

dataset = pd.read_csv('train.csv')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\W+', ' ', text)

    return text 

dataset['text'] = dataset['text'].apply(clean_text)
text = dataset["text"].tolist()
target = dataset["target"].tolist()
x_train, x_eval, y_train, y_eval = train_test_split(text, target, test_size=0.2)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf_vect = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = tfidf_vect.fit_transform(x_train)
y_train = np.array(y_train)

X_train_tfidf.shape

(6090, 65102)

In [3]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression().fit(X_train_tfidf, y_train)

X_eval_tfidf = tfidf_vect.transform(x_eval)
y_eval = np.array(y_eval)
eval_predicted = clf.predict(X_eval_tfidf)

from sklearn import metrics

print(metrics.classification_report(y_eval, eval_predicted))
metrics.confusion_matrix(y_eval, eval_predicted)

              precision    recall  f1-score   support

           0       0.78      0.90      0.83       856
           1       0.84      0.66      0.74       667

    accuracy                           0.80      1523
   macro avg       0.81      0.78      0.79      1523
weighted avg       0.81      0.80      0.79      1523



array([[774,  82],
       [224, 443]], dtype=int64)

In [4]:
test = pd.read_csv('test.csv')

test['text'] = test['text'].apply(clean_text)
test_text = test["text"].tolist()

X_test_tfidf = tfidf_vect.transform(test_text)
test_predicted = clf.predict(X_test_tfidf)

# 将预测结果添加到 submission.csv
df = pd.DataFrame(test_predicted, columns=['target'])
submission_logic = pd.DataFrame({'id': test['id'], 'target': test_predicted})

# 保存结果
submission_logic.to_csv('submission_logic.csv', index=False)